# Prepare Salcher atlas reference subset for CosMx cell typing

**Goal:** Subset the Salcher et al. pan-cancer scRNA-seq atlas to the genes present
in the CosMx 1000-plex panel, creating pseudo-probes for multi-gene probes by
summing raw counts of individual gene targets. The resulting object has column
names matching CosMx display names exactly, enabling direct comparison between
the two datasets.

**Input:** Full Salcher atlas (`salcher_atlas.raw`); QC-filtered CosMx AnnData
(`cosmx.processed/combined_adata_qc_filtered.h5ad`) for the CosMx gene list;
Bruker panel gene list (`cosmx.gene_conversion_list`) for multi-gene probe mapping.

**Output:** `salcher_atlas.cosmx_subset` — Salcher atlas with 975 pseudo-probe
columns matching CosMx display names, raw counts summed for multi-gene probes;
`cosmx.gene_annotations` — CSV documenting CosMx genes absent from the Salcher
reference with biological annotations.

## Setup

In [ ]:
from pathlib import Path
import yaml

import pandas as pd
import anndata
import scanpy as sc
import seaborn as sns

sns.set(font_scale=1.8)
sns.set_style('ticks')

In [ ]:
# load paths from central config
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
salcher_path        = Path(cfg['datasets']['salcher_atlas']['raw'])
cosmx_filtered_path = repo_root / cfg['outputs']['processed'] / 'combined_adata_qc_filtered.h5ad'
gene_probe_path     = repo_root / cfg['datasets']['cosmx']['gene_conversion_list']

# Bruker panel gene list — maps CosMx display names to gene symbols
gene_probe_path = repo_root / cfg['datasets']['cosmx']['gene_conversion_list']

# outputs
salcher_subset_path   = Path(cfg['datasets']['salcher_atlas']['cosmx_subset'])
gene_annotations_path = repo_root / cfg['datasets']['cosmx']['gene_annotations']


## Load data

In [ ]:
# load full Salcher atlas
print("Loading Salcher atlas...")
ref_adata = sc.read_h5ad(salcher_path)
print(f"Salcher atlas: {ref_adata.n_obs:,} cells × {ref_adata.n_vars:,} genes")

# rename var_names from Ensembl IDs to gene symbols for matching
# note: anndata warns that var_names are categorical strings after renaming from
# Ensembl IDs to gene symbols — this is expected and does not affect functionality
ref_adata.var_names = ref_adata.var['feature_name'].values
ref_genes = set(ref_adata.var_names)

# load CosMx QC-filtered data to get the panel gene list
print("Loading CosMx data...")
adata = anndata.read_h5ad(cosmx_filtered_path)
print(f"CosMx panel: {adata.n_vars} genes")

# load Bruker panel gene list
gene_conversion_list = pd.read_csv(gene_probe_path, skiprows=1)
print(f"Panel file columns: {gene_conversion_list.columns.tolist()}")

## Gene overlap analysis

Of the 1000 genes in the CosMx panel, 975 are matched to the Salcher atlas.
Single-gene probes are matched directly by gene symbol. Multi-gene probes
(probes targeting multiple gene family members) are resolved to their individual
targets present in the Salcher atlas — one target per probe, except HLA-DQB1/2
where both subunits are retained given their relevance to MHC II biology.

Genes absent from the Salcher atlas are documented with biological annotations
in `cosmx_genes_not_in_salcher_reference.csv`. Most are tissue-specific genes
not expected in lung, lncRNAs with naming mismatches, or multi-gene probes
with no matching targets in the reference.

In [ ]:
from collections import Counter

# filter out non-gene rows (negative controls, system controls, copyright notice)
valid_rows = gene_conversion_list[
    ~gene_conversion_list['Display Name'].str.contains(
        'Negative|System|©|reserved', na=False, regex=True)
]

misnamed_genes = {'BBLN': 'C9orf16', 'RGS5': 'RGS5_ENSG00000143248'}

genes_to_keep = []
multi_gene     = []
leftovers      = []

for _, row in valid_rows.iterrows():
    name = row['Display Name']

    if name in ['IFNA1/13', 'IFNL2/3']:
        leftovers.append(name)
    elif '/' in name or 'MHC' in name or name in ['HLA-DQA1', 'HLA-DRB']:
        multi_gene.append(name)
    elif name in ref_genes:
        genes_to_keep.append(name)
    elif name in misnamed_genes and misnamed_genes[name] in ref_genes:
        genes_to_keep.append(misnamed_genes[name])
    else:
        leftovers.append(name)

print(f"Single-gene matches: {len(genes_to_keep)}")
print(f"Multi-gene probes:   {len(multi_gene)}")
print(f"Leftovers:           {len(leftovers)}")

## Build multi-gene probe pseudo-probes

For each multi-gene probe (e.g. `MHC I`, `HLA-DQB1/2`), the raw counts of all
individual gene targets present in the Salcher atlas are summed to create a
single pseudo-probe column. This mirrors what the CosMx instrument does physically
— a single probe hybridizes to multiple related transcripts — and ensures column
names in the reference match CosMx display names exactly.

Raw counts from `layers['count']` are used rather than normalized values so that
the summed pseudo-probe values are on the same scale as single-gene probes.

In [ ]:
# set index to display name for easy lookup
gene_conversion_list['New Index'] = gene_conversion_list['Display Name']
gene_conversion_list = gene_conversion_list.set_index('New Index')

# build dict mapping each individual gene target to its CosMx probe display name
multi_genes_probes_big_list = []
multi_genes_targets_big_list = []

for index, row in gene_conversion_list.iterrows():
    if '|' in str(gene_conversion_list.loc[index, 'Gene Symbol(s)']):
        multi_genes_probes_big_list.append(gene_conversion_list.loc[index, 'Display Name'])
        parts = row['Gene Symbol(s)'].split(' | ')
        for part in parts:
            if part not in multi_genes_targets_big_list:
                multi_genes_targets_big_list.append(part)

# map each individual gene target back to its probe display name
multi_gene_dict = {}
for j in multi_genes_targets_big_list:
    for i in multi_genes_probes_big_list:
        if j in gene_conversion_list.loc[i, 'Gene Symbol(s)']:
            multi_gene_dict[j] = i

print(f"Multi-gene targets found: {len(multi_gene_dict)}")

# remove targets not in Salcher atlas
not_in_dataset = [g for g in multi_gene_dict if g not in ref_genes]
for g in not_in_dataset:
    multi_gene_dict.pop(g)

print(f"Multi-gene targets in Salcher atlas: {len(multi_gene_dict)}")

In [ ]:
# map gene symbols to Ensembl IDs for indexing into ref_adata
# note: ref_adata var_names were renamed to symbols above, so we use the
# original Ensembl IDs stored in the index before renaming
# reload original var to get Ensembl → symbol mapping
ref_adata_orig = sc.read_h5ad(salcher_path)  # reload to get original Ensembl IDs

gene_converter_to_ens = {}
gene_converter_from_ens = {}
ens_multi_gene_names = []

for k in multi_gene_dict.keys():
    ens = ref_adata_orig.var[ref_adata_orig.var['feature_name'] == k].index[0]
    ens_multi_gene_names.append(ens)
    gene_converter_to_ens[k] = ens
    gene_converter_from_ens[ens] = k

print(f"Ensembl IDs mapped: {len(ens_multi_gene_names)}")

In [ ]:
%%time

# extract raw counts for multi-gene targets from Salcher atlas
expression_data = ref_adata_orig[:, ens_multi_gene_names].layers['count'].toarray()
cell_names = ref_adata_orig.obs_names.tolist()

multi_gene_df = pd.DataFrame(expression_data, index=cell_names, columns=ens_multi_gene_names)

# sum raw counts per probe display name to create pseudo-probes
def sum_multi_gene_probe(df, columns):
    """Sum raw counts across individual gene targets for a multi-gene probe."""
    return df[columns].sum(axis=1)

unique_probes = list(dict.fromkeys(multi_gene_dict.values()))
multi_sum_df = pd.DataFrame(index=cell_names)

for probe in unique_probes:
    # get all gene targets for this probe
    targets = [g for g, p in multi_gene_dict.items() if p == probe]
    ens_targets = [gene_converter_to_ens[g] for g in targets if g in gene_converter_to_ens]
    multi_sum_df[probe] = sum_multi_gene_probe(multi_gene_df, ens_targets)

print(f"Pseudo-probes created: {multi_sum_df.shape[1]}")
print(multi_sum_df.sum(axis=1).describe())

## Build single-gene matrix

Raw counts for single-gene probes are extracted from the Salcher atlas using
Ensembl IDs, then renamed to CosMx display names for consistency. Misnamed
genes (`BBLN` → `C9orf16`, `RGS5` → `RGS5_ENSG00000143248`) are handled here.

In [ ]:
%%time

# genes_to_keep contains single-gene CosMx display names matched to Salcher
# remove any that overlap with multi-gene probe targets to avoid duplication
genes_to_query = [g for g in genes_to_keep if g not in multi_sum_df.columns]

# map to Ensembl IDs
ens_single = []
ens_to_display = {}
display_to_ens = {}

for g in genes_to_query:
    # handle misnamed genes
    lookup = misnamed_genes.get(g, g)
    try:
        ens = ref_adata_orig.var[ref_adata_orig.var['feature_name'] == lookup].index[0]
        ens_single.append(ens)
        ens_to_display[ens] = g
        display_to_ens[g] = ens
    except IndexError:
        print(f"Warning: {g} ({lookup}) not found in Salcher var")

print(f"Single-gene Ensembl IDs mapped: {len(ens_single)}")

# extract raw counts
expression_data = ref_adata_orig[:, ens_single].layers['count'].toarray()
gene_display_names = [ens_to_display[e] for e in ens_single]

single_gene_df = pd.DataFrame(
    expression_data,
    index=cell_names,
    columns=gene_display_names,
)

print(f"Single-gene matrix: {single_gene_df.shape}")

## Document missing genes

Genes in the CosMx panel with no match in the Salcher atlas are documented
below with biological annotations. Most are tissue-specific (cardiac, pancreatic,
mammary) and not expected in lung adenocarcinoma, or lncRNAs with naming
conventions that differ between the two datasets.

In [ ]:
leftover_dict = {
    # tissue-specific genes not expected in lung
    'ADIPOQ':    'AdipoQ — adipokine, not in Salcher atlas',
    'ATP5F1E':   'ATP synthase F1 subunit epsilon — not in atlas',
    'CALB1':     'Calbindin 1 — not in atlas',
    'CIDEA':     'Cell death-inducing DFFA-like effector A — mammary-specific; CIDEB and CIDEC present',
    'CRP':       'C-reactive protein — not in atlas',
    'CSHL1':     'Chorionic somatomammotropin hormone-like 1 — mammary-related',
    'GC':        'Vitamin D-binding protein — not in atlas',
    'GCG':       'Glucagon — not expected in lung',
    'GPX1':      'Glutathione peroxidase 1 — GPX 2,3,4,7,8 present',
    'INS':       'Insulin — not expected in lung',
    'KLK3':      'Prostate-specific antigen — not expected in lung',
    'KRT20':     'Keratin 20 — GI-specific; 18 other keratin genes present',
    'MYH6':      'Myosin heavy chain alpha — cardiac muscle specific',
    'MYL7':      'Atrial myosin regulatory light chain — cardiac atria specific',
    'PRTN3':     'Proteinase 3 — mainly neutrophil granulocytes',
    'PTPRCAP':   'Protein tyrosine phosphatase type C-associated protein — not in atlas',
    'QRFPR':     'Pyroglutamylated RFamide peptide receptor — mainly brain',
    'REG1A':     'Pancreatic stone protein — not in atlas',
    'SST':       'Somatostatin — not in atlas',
    'TTR':       'Transthyretin — not in atlas',
    'VEGFD':     'C-fos-induced growth factor — VEGFA, B, C present',
    # lncRNAs — naming convention mismatch between CosMx and Salcher
    'LINC01781': 'Long non-coding RNA — lncRNA naming not in reference',
    'LINC01857': 'Long non-coding RNA — lncRNA naming not in reference',
    'LINC02446': 'Long non-coding RNA — lncRNA naming not in reference',
    # gene aliases handled via misnamed_genes dict
    'BBLN':      'Alias for C9orf16 — present in atlas under canonical name',
    'RGS5':      'Present in atlas as RGS5_ENSG00000143248',
    # multi-gene probes with no targets in reference
    'IFNA1/13':  'Multi-gene probe — IFN-alpha subtypes; naming convention not in reference',
    'IFNL2/3':   'Multi-gene probe — IFN-lambda subtypes; naming convention not in reference',
}

leftover_df = pd.DataFrame.from_dict(leftover_dict, orient='index', columns=['annotation'])
leftover_df.index.name = 'gene'
leftover_df.to_csv(gene_annotations_path)
print(f"Saved gene annotations ({len(leftover_df)} genes) → {gene_annotations_path}")
print(leftover_df)

## Combine and save

Single-gene and multi-gene pseudo-probe matrices are concatenated into a single
DataFrame and converted to AnnData. Column names match CosMx display names
exactly — enabling direct gene-name matching in `umap_cosmx` and `cosmx_cell_typing`
without any additional remapping.

In [ ]:
# concatenate single and multi-gene matrices
ref_df = pd.concat([single_gene_df, multi_sum_df], axis=1)
print(f"Combined reference matrix: {ref_df.shape}")


In [ ]:
# rename misnamed genes in the dataframe before building AnnData
rename_map = {
    'C9orf16': 'BBLN',
    'RGS5_ENSG00000143248': 'RGS5',
}
ref_df = ref_df.rename(columns=rename_map)

# verify column names match CosMx display names
cosmx_display_names = set(adata.var_names)
matched = [c for c in ref_df.columns if c in cosmx_display_names]
print(f"Columns matching CosMx display names: {len(matched)}")


In [ ]:
# build AnnData with CosMx display names as var_names
ref_cosmx = anndata.AnnData(
    X=ref_df.values,
    obs=ref_adata_orig.obs.copy(),
)
ref_cosmx.var_names = ref_df.columns.tolist()

print(f"BBLN in ref: {'BBLN' in ref_cosmx.var_names}")
print(f"RGS5 in ref: {'RGS5' in ref_cosmx.var_names}")
print(f"Final reference AnnData: {ref_cosmx.n_obs:,} cells × {ref_cosmx.n_vars} genes")

ref_cosmx.write(salcher_subset_path)
print(f"Saved → {salcher_subset_path}")